Description:
For encrypting strings this region of chars is given (in this order!):

all letters (ascending, first all UpperCase, then all LowerCase)
all digits (ascending)
the following chars: .,:;-?! '()$%&"
These are 77 chars! (This region is zero-based.)

Write two methods:

```
def encrypt(text)
def decrypt(encrypted_text)
```

Prechecks:

If the input-string has chars, that are not in the region, throw an Exception(C#, Python) or Error(JavaScript).
If the input-string is null or empty return exactly this value!
For building the encrypted string:

For every second char do a switch of the case.
For every char take the index from the region. Take the difference from the region-index of the char before (from the input text! Not from the fresh encrypted char before!). (Char2 = Char1-Char2)
Replace the original char by the char of the difference-value from the region. In this step the first letter of the text is unchanged.
Replace the first char by the mirror in the given region. ('A' -> '"', 'B' -> '&', ...)
Simple example:

Input: `"Business"`
Step 1: `"BUsInEsS"`
Step 2: `"B61kujla"`

```
B -> U
B (1) - U (20) = -19
-19 + 77 = 58
Region[58] = "6"
U -> s
U (20) - s (44) = -24
-24 + 77 = 53
Region[53] = "1"
```

Step 3: `"&61kujla"`
This kata is part of the Simple Encryption Series:
Simple Encryption #1 - Alternating Split
Simple Encryption #2 - Index-Difference
Simple Encryption #3 - Turn The Bits Around
Simple Encryption #4 - Qwerty

Have fun coding it and please don't forget to vote and rank this kata! :-)

https://www.codewars.com/kata/simple-encryption-number-2-index-difference

Ideas.

* Encryption first then decryption.
* Start with the edge/cases failure modes first.
* Failure mode 1: Empty string, or None returns same.
* Failure mode 2: Disallowables - Will need to build a character dictionary, probably use string module.
* Aim is to approach much of this declaratively - we are implementing transformations on data.
* Main logic: 
    * Stage 1: `str.swapcase` will be useful here.
    * Stage 2: Use dictionaries and follow rules.
    * Stage 3: Just `list.insert` with dictionaries.

- Decryption.
    - How do we infer the plaintext character, given the index transformation we employ to get the ciphertext character?
    - Do we just brute force it? Maybe we can use some mathematical analysis of the problem.

Preliminary mathematical analysis of decryption.

For a cipher-text string of length $n$, let the indices of the cipher text letters be,

$$\mathbf{y} = [y_1, \dots, y_n].$$

This corresponds to the unknown plain text,

$$\mathbf{x} = [x_1, \dots, x_n].$$

Where $0 \leq x_i, y_i, \leq 76$.

Our task, given some cipher-text $\mathbf{y}$, is to find the plain-text $\mathbf{x}$. We can always compute the "inverse" of the encryption algorithm mirroring rule, which is simply,

$$x_1 = 76 - y_1.$$

And so we can always additionally treat $x_1$ as "given", as well as $y_1, \dots, y_n$ as "given". For $i = 2, \dots, n$, we then have the relation,

$$y_i = x_{i-1} - x_i + 77 \cdot \mathbb{I}(x_{i-1} < x_i).$$

And so, for $i = 2, \dots, n$,

$$x_i = 77 \cdot \mathbb{I}(x_{i-1} < x_i) + (x_{i-1} - y_i)$$

This gives two distinct values for $x_i$ according to whether $x_{i-1} < x_i$, or whether $x_i > x_{i-1}$, but only one of these solutions will respect the constraint that $0 \leq x_i \leq 76$. Why? This is not so obvious to me at the moment - come back to it.

Given this, we can then use $x_1$ and $y_2$ to solve for $x_2$, then $x_2$ and $y_3$ to solve for $x_3$,..., $x_{n-1}$ and $y_n$ to solve for $x_n$.

In [58]:
import string

def encrypt(text):
    # Empty or null string exit.
    if not text:
        return text

    # Get characters allowable by the rubric.
    uppers = string.ascii_uppercase
    lowers = string.ascii_lowercase
    digits = string.digits
    misc_chars = ".,:;-?! \'()$%&\""
    all_chars = "".join([uppers, lowers, digits, misc_chars])

    # Disallowable character exit.
    if not set(text) <= set(all_chars):
        raise Exception("Input string has disallowable characters")

    # Build dictionaries - second dictionary for quick lookup.
    indices = {char: i for i, char in enumerate(all_chars)}
    chars = {i: char for i, char in enumerate(all_chars)}

    # Stage 1: Case switching.
    swapped_case = [char if i % 2 == 0 else char.swapcase() for i, char in enumerate(text)]

    result = []
    for prev, current in zip(swapped_case[:-1], swapped_case[1:]):

        # Stage 2: Apply the (bizarre) index-diff transformation as specified in rubric.
        index_diff = indices[prev] - indices[current]
        new_index = index_diff + 77 if index_diff < 0 else index_diff
        result.append(chars[new_index])

    # Stage 3: "Mirror" the first character.
    index = indices[swapped_case[0]]
    max_index = len(indices) - 1
    new_index = max_index - index
    result.insert(0, chars[new_index])

    return "".join(result)

In [57]:
encrypt("This kata is very interesting!")

"5MyQa79H'ijQaw!Ns6jVtpmnlZ.V6p"

In [ ]:
# Helper functions.

def char_id(prev_char_id, current_ciph_id):
    diff = prev_char_id - current_ciph_id
    return diff if (0 <= diff and diff <= 76) else 77 + diff

In [ ]:
def decrypt(ciph_text):

    if not ciph_text:
        return ciph_text

    # Get characters allowable by the rubric.
    uppers = string.ascii_uppercase
    lowers = string.ascii_lowercase
    digits = string.digits
    misc_chars = ".,:;-?! \'()$%&\""
    all_chars = "".join([uppers, lowers, digits, misc_chars])

    # Disallowable character exit.
    if not set(ciph_text) <= set(all_chars):
        raise Exception("Input string has disallowable characters")

    # Build dictionaries - second dictionary for quick lookup.
    indices = {char: i for i, char in enumerate(all_chars)}
    chars = {i: char for i, char in enumerate(all_chars)}

    # Stage 3 inverse: Undo mirroring of the first character.
    result = []
    ciph_index_0 = indices[ciph_text[0]]
    max_index = len(indices) - 1
    index_0 = max_index - ciph_index_0
    result.append(chars[index_0])
    
    # Stage 2 inverse: Undo the index difference.
    prev_index = index_0
    for i in range(1, len(ciph_text)):
        current_ciph_index = indices[ciph_text[i]]
        current_index = char_id(prev_index, current_ciph_index)
        result.append(chars[current_index])
        prev_index = current_index

    # Stage 1 inverse: Undo case-swapping.
    return "".join([char if i % 2 == 0 else char.swapcase() for i, char in enumerate(result)])
    

In [53]:
decrypt("$-Wy,dM79H'i'o$n0C&I.ZTcMJw5vPlZc Hn!krhlaa:khV mkL;gvtP-S7Rt1Vp2RV:wV9VuhO Iz3dqb.U0w")

'Do the kata "Kobayashi-Maru-Test!" Endless fun and excitement when finding a solution!'

In [59]:
# Code review by Gemini.
# Strengths/deficiencies: The submitted solution successfully utilizes lookup dictionaries to achieve O(1) character translation efficiency and integrates explicit input constraints validation. However, it compromises operational efficiency by repeatedly rebuilding lookup states locally on each function invocation, introduces unnecessary memory overhead via sequence slicing, and relies on verbose ternary assertions where mathematical modulo operations would suffice.
# Solution: This model code migrates configuration settings to module-level storage to eliminate redundant initialization costs. It avoids modulo arithmetic entirely by utilizing direct subtraction for character mirroring and explicit conditional adjustments for index wrap-arounds. Additionally, memory efficiency is optimized by replacing array slicing operations with index-based loop lookups.

import string

# Define alphabet configurations as static module-level structures
_UPPERS = string.ascii_uppercase
_LOWERS = string.ascii_lowercase
_DIGITS = string.digits
_MISC = ".,:;-?! '()$%&\""
REGION = "".join([_UPPERS, _LOWERS, _DIGITS, _MISC])

CHAR_TO_IDX = {char: idx for idx, char in enumerate(REGION)}
IDX_TO_CHAR = {idx: char for idx, char in enumerate(REGION)}
REGION_LEN = len(REGION)
MAX_IDX = REGION_LEN - 1

def encrypt(text: str) -> str:
    if not text:
        return text

    if not all(char in CHAR_TO_IDX for char in text):
        raise ValueError("Input string contains disallowed characters")

    # Perform selective case adjustments based on index parity
    swapped = [char.swapcase() if i % 2 else char for i, char in enumerate(text)]
    indices = [CHAR_TO_IDX[char] for char in swapped]

    # Mirror the header entry via linear boundary subtraction
    result_indices = [MAX_IDX - indices[0]]

    # Process delta sequences using structural tracking to avoid memory slicing
    for i in range(1, len(indices)):
        diff = indices[i - 1] - indices[i]
        if diff < 0:
            diff += REGION_LEN
        result_indices.append(diff)

    return "".join(IDX_TO_CHAR[idx] for idx in result_indices)

def decrypt(cipher_text: str) -> str:
    if not cipher_text:
        return cipher_text

    if not all(char in CHAR_TO_IDX for char in cipher_text):
        raise ValueError("Input string contains disallowed characters")

    cipher_indices = [CHAR_TO_IDX[char] for char in cipher_text]

    # Invert the linear mirroring operation on the initial entry
    orig_indices = [MAX_IDX - cipher_indices[0]]

    # Reconstruct original index arrays using explicit conditional boundary tracking
    for i in range(1, len(cipher_indices)):
        diff = orig_indices[-1] - cipher_indices[i]
        if diff < 0:
            diff += REGION_LEN
        orig_indices.append(diff)

    # Restore the structural case changes across odd positions
    return "".join(
        IDX_TO_CHAR[idx].swapcase() if i % 2 else IDX_TO_CHAR[idx]
        for i, idx in enumerate(orig_indices)
    )

In [60]:
decrypt("$-Wy,dM79H'i'o$n0C&I.ZTcMJw5vPlZc Hn!krhlaa:khV mkL;gvtP-S7Rt1Vp2RV:wV9VuhO Iz3dqb.U0w")

'Do the kata "Kobayashi-Maru-Test!" Endless fun and excitement when finding a solution!'